# 概率与信息论基础

> 词表只有「的、是、在、了」四个 Token 时，模型可能给出概率 `[0.3, 0.2, 0.4, 0.1]`。每个数位于 0 到 1 之间，总和等于 1；模型输出的首先是一个概率分布，而不是确定答案。
>
> 从分数到选择：**Softmax** 把 logits 转换为概率，采样再按照概率选出一个 Token。
>
> 从概率到训练：**Cross-Entropy** 衡量模型分布与正确答案之间的差距，其梯度直接决定 logits 应该怎样调整。
>
> 从分布到度量：**Entropy** 描述不确定性，**KL Divergence** 比较两个分布，**Perplexity** 把平均 Cross-Entropy 转换为更直观的尺度。这套数学也会复用于 MoE Router 与模型对齐。


## 1. 先搞懂：模型输出的「概率」到底是什么

假设词表里有 V 个词。模型在每个位置给出的不是一个词，而是一组概率 $p = (p_1, \ldots, p_V)$，满足两件事：每个概率都不小于 0，而且加起来等于 1。

这种「在有限个类别上各给一个概率」的东西，就叫 categorical distribution（类别分布）。别被名字吓到。它其实就是「在几个选项里各占多少比例」。

举个例子，V=4，模型给出 logits（下面马上解释这个词）为 $z = [2.0, 1.0, 0.1, -1.0]$。这组数还不是概率，但它唯一确定了一个 categorical distribution。

你可能已经在别处见过它的另一个名字：MoE 一节会讲 expert routing，那里做的也是同一件事——给每个专家一个概率，看这个 token 归谁。所以「预测下一个词」和「选专家」在数学上是同一件事。

In [1]:
import math
import torch
import torch.nn.functional as F

torch.manual_seed(42)

# 用一组极小数字演示 categorical 分布
logits = torch.tensor([2.0, 1.0, 0.1, -1.0])
print("logits:           ", logits.tolist())
print("指数化 exp(logits):", [round(x, 4) for x in torch.exp(logits).tolist()])
print("归一化后概率:      ", [round(x, 4) for x in F.softmax(logits, dim=-1).tolist()])
print("概率之和:          ", round(F.softmax(logits, dim=-1).sum().item(), 6))
print()
print("关键观察：logits 是任意实数，softmax 把它压成一组非负、和为 1 的概率。")
print("这正是 categorical 分布的参数化形式：V 个类别，每个类别一个概率。")

logits:            [2.0, 1.0, 0.10000000149011612, -1.0]
指数化 exp(logits): [7.3891, 2.7183, 1.1052, 0.3679]
归一化后概率:       [0.6381, 0.2347, 0.0954, 0.0318]
概率之和:           1.0

关键观察：logits 是任意实数，softmax 把它压成一组非负、和为 1 的概率。
这正是 categorical 分布的参数化形式：V 个类别，每个类别一个概率。


### 1.1 从 logits 到概率：softmax 这个转换器

logits 是模型输出的原始分数，可以是任意实数——正数、负数、大数、小数都行。概率却要求非负、和为 1。谁来做这个转换？softmax。$$p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$拆开看：指数 $e^{z_i}$ 永远为正，保证分子非负；分母把所有 $e^{z_j}$ 加起来，一除就归一化到和为 1。而且它保留了大小关系：$z_i > z_j$ 推出 $p_i > p_j$。所以「分数最高的词」和「概率最高的词」是同一个——这就是 greedy decoding 直接取最大分数就行的原因。一个问题：为什么偏偏用指数，不能是别的？因为 softmax 是唯一能让「最大似然」等价于「最小化 cross-entropy」的可微函数（后面讲 loss 的一节会用到这个结论）。这是 LM 训练能用 CE loss 的数学根源。



### 1.2 一个小坑：指数会溢出

直接按定义算 softmax 有个隐患。当 $z_i$ 很大时（比如 1000），$e^{1000}$ 直接溢出成 inf。这不是理论问题，是实际会踩的坑。解决办法用一个聪明性质：softmax 对「整体平移」不变。给所有 $z_i$ 同时减去同一个数 $c$，结果不变：$$\text{softmax}(z)_i = \frac{e^{z_i - c}}{\sum_j e^{z_j - c}}$$取 $c = \max_j z_j$，最大的那个指数变成 $e^0 = 1$，其余的都在 1 以内，不会溢出。更进一步，把 $e$ 和 $\log$ 一起算，得到 logsumexp：$$\text{logsumexp}(z) = c + \log \sum_j e^{z_j - c}$$于是数值稳定的 log-softmax 就是 $\log p_i = z_i - \text{logsumexp}(z)$。这条式子是 FlashAttention 里 online softmax 的起点——后面附录 D 会用到。PyTorch 的 `F.log_softmax` 和 `F.cross_entropy` 内部都这么做了，所以平时不用从零实现，但读懂它是读推理框架 kernel 的前提。



In [2]:
# 对比：朴素 softmax vs 数值稳定 softmax
big_logits = torch.tensor([1000.0, 1001.0, 1002.0])

naive_exp = torch.exp(big_logits)
print("朴素实现：")
print("  exp([1000, 1001, 1002]) =", naive_exp.tolist())
print("  → 全部溢出为 inf，无法归一化")
print()

# 减去最大值再做
stable_shifted = big_logits - big_logits.max()
stable_exp = torch.exp(stable_shifted)
stable_probs = stable_exp / stable_exp.sum()
print("数值稳定版（减去 max=1002）：")
print("  shifted logits =", stable_shifted.tolist())
print("  exp(shifted)   =", [round(x, 6) for x in stable_exp.tolist()])
print("  softmax        =", [round(x, 4) for x in stable_probs.tolist()])
print()

# PyTorch 内部已经这么做了
torch_logsoftmax = F.log_softmax(big_logits, dim=-1)
print("PyTorch log_softmax:", [round(x, 4) for x in torch_logsoftmax.tolist()])
print("关键观察：减去 max 不改变 softmax 结果，但避免 exp 溢出。")

朴素实现：
  exp([1000, 1001, 1002]) = [inf, inf, inf]
  → 全部溢出为 inf，无法归一化

数值稳定版（减去 max=1002）：
  shifted logits = [-2.0, -1.0, 0.0]
  exp(shifted)   = [0.135335, 0.367879, 1.0]
  softmax        = [0.09, 0.2447, 0.6652]

PyTorch log_softmax: [-2.4076, -1.4076, -0.4076]
关键观察：减去 max 不改变 softmax 结果，但避免 exp 溢出。


## 2. 采样：从「概率」到「真选一个词」

推理时，模型要从 categorical distribution 里抽一个词。这件事在概率论里叫「采样」（sampling）。上一节讲的 temperature、top-k、top-p，本质上都是先改一改这个分布，再采样。这些参数在解码策略一节讲过，把它们放回概率语言里，工程参数就有了精确含义。下面用同一组 logits 演示三种采样怎么改变输出分布。



### 2.1 Temperature：一个「大胆还是保守」的旋钮

Temperature 采样的公式是 $p^{(T)} = \text{softmax}(z / T)$。注意：除的是 logits，不是概率。这一步很关键——温度作用在「未归一化的分数」上，而不是直接调概率。看两个极端，你就明白它为什么管用：- $T \to 0^+$：$z/T$ 越除越大，最大那一项的指数把其他全压下去，softmax 输出几乎变成 one-hot。等价于 greedy（只选最确定的）。- $T \to \infty$：$z/T \to 0$，所有 $e^{z_i/T} \to 1$，softmax 输出趋向均匀分布。等价于「在所有词上瞎猜」。T=1 时分布和原始 softmax 完全一致。所以 T 是一个连续旋钮：拧到 0 是「我就要最确定的」，拧到无穷是「随便哪个都行」。

In [3]:
logits = torch.tensor([2.0, 1.0, 0.1, -1.0])
print(f"{'T':>5}  {'p0':>7}  {'p1':>7}  {'p2':>7}  {'p3':>7}  说明")
print("-" * 70)

for T in [0.1, 0.5, 1.0, 2.0, 10.0, 1000.0]:
    probs = F.softmax(logits / T, dim=-1)
    row = f"{T:>5.1f}  " + "  ".join(f"{p:>7.4f}" for p in probs.tolist())
    if T <= 0.1:
        note = "接近 argmax（greedy）"
    elif T == 1.0:
        note = "原始分布"
    elif T >= 1000.0:
        note = "接近 uniform"
    else:
        note = ""
    print(row + "  " + note)

print()
print("关键观察：T 是连续旋钮。T→0 把分布压成 one-hot，T→∞ 拉成 uniform。")

    T       p0       p1       p2       p3  说明
----------------------------------------------------------------------
  0.1   1.0000   0.0000   0.0000   0.0000  接近 argmax（greedy）
  0.5   0.8619   0.1166   0.0193   0.0021  
  1.0   0.6381   0.2347   0.0954   0.0318  原始分布
  2.0   0.4512   0.2737   0.1745   0.1007  
 10.0   0.2880   0.2606   0.2381   0.2133  
1000.0   0.2504   0.2501   0.2499   0.2496  接近 uniform

关键观察：T 是连续旋钮。T→0 把分布压成 one-hot，T→∞ 拉成 uniform。


### 2.2 Top-k：只在一小撮里选

Top-k 的思路更直白：先把概率最高的 k 个词挑出来，其余的一律排除，然后在剩下的 k 个里重新归一。写成公式：设 $S_k$ 是概率最高的 k 个词的编号集合，则$$p^{(\text{top-}k)}_i = \begin{cases} \frac{p_i}{\sum_{j \in S_k} p_j} & i \in S_k \\ 0 & i \notin S_k \end{cases}$$k=1 时只剩一个词，归一化后必然 One-hot，等于 greedy。工程实现里通常不真的置 0，而是把被排除位置的 logits 改成 $-\infty$，softmax 后自然变成 0。这和 attention 里 mask 用 `-inf` 是同一个技巧。

### 2.3 Top-p：别写死个数，看累积概率

Top-k 有个毛病：k 是写死的。但不同位置的概率分布形状差很多——分布很尖的时候 k=40 显得太多，分布很平的时候 k=40 又太少。Top-p 改成按「累积概率」来截断，k 不再是固定值。先把所有词按概率从大到小排序，得 $p_{(1)} \ge p_{(2)} \ge \ldots$。Top-p 选一个最小的前缀集合 $S_p$，让这些概率加起来刚好达到 $p$（比如 0.9）。集合之外的词一起置 0 再归一化。p=1 时什么都不截断（纯采样），p 很小就退化成 greedy。有个工程细节：HuggingFace 会把「累积概率刚刚超过 p」的那个词**也保留**，所以实际保留的累积概率可能略大于 p。下面代码用同一组 logits 演示三种方式的差异。

In [4]:
# 用一组较尖锐的 logits 演示三种采样的输出差异
logits = torch.tensor([3.0, 2.0, 1.0, 0.5, 0.0, -1.0])
base_probs = F.softmax(logits, dim=-1)
print("原始 logits:", logits.tolist())
print("原始概率:  ", [round(p, 4) for p in base_probs.tolist()])
print()

# Temperature: T=0.5
probs_t = F.softmax(logits / 0.5, dim=-1)
print("Temperature=0.5（低温，更尖锐）:")
print("  ", [round(p, 4) for p in probs_t.tolist()])
print()

# Top-k=2: 保留前 2 大
top2_vals, top2_idx = torch.topk(logits, 2)
mask_k = torch.full_like(logits, float('-inf'))
mask_k[top2_idx] = top2_vals
probs_k = F.softmax(mask_k, dim=-1)
print("Top-k=2（截断到 2 个）:")
print("  ", [round(p, 4) if p > 0 else '  0   ' for p in probs_k.tolist()])
print()

# Top-p=0.8: 累积概率达到 0.8
sorted_probs, sorted_idx = torch.sort(base_probs, descending=True)
cum = torch.cumsum(sorted_probs, dim=0)
keep_mask = cum <= 0.8
keep_mask[0] = True  # 至少保留一个
# 把累积值刚刚超过 0.8 的那个 token 也保留
first_exceed = (cum > 0.8).nonzero()
if len(first_exceed) > 0:
    keep_mask[first_exceed[0].item()] = True
kept_idx = sorted_idx[keep_mask]
mask_p = torch.full_like(logits, float('-inf'))
mask_p[kept_idx] = logits[kept_idx]
probs_p = F.softmax(mask_p, dim=-1)
print("Top-p=0.8（累积概率截断）:")
print("  ", [round(p, 4) if p > 0 else '  0   ' for p in probs_p.tolist()])
print()
print("关键观察：")
print("  Temperature 改变所有概率的相对比例，但不置零任何类别")
print("  Top-k 按固定数量截断，Top-p 按累积概率截断")
print("  工业默认是 Top-k + Top-p 同时使用，先固定上限再自适应")

原始 logits: [3.0, 2.0, 1.0, 0.5, 0.0, -1.0]
原始概率:   [0.6048, 0.2225, 0.0819, 0.0496, 0.0301, 0.0111]

Temperature=0.5（低温，更尖锐）:
   [0.8597, 0.1163, 0.0157, 0.0058, 0.0021, 0.0003]

Top-k=2（截断到 2 个）:
   [0.7311, 0.2689, '  0   ', '  0   ', '  0   ', '  0   ']

Top-p=0.8（累积概率截断）:
   [0.7311, 0.2689, '  0   ', '  0   ', '  0   ', '  0   ']

关键观察：
  Temperature 改变所有概率的相对比例，但不置零任何类别
  Top-k 按固定数量截断，Top-p 按累积概率截断
  工业默认是 Top-k + Top-p 同时使用，先固定上限再自适应


### 2.4 补充：Gumbel-Max trick，采样还能这么写

从 categorical distribution 采样还有另一种等价写法：给每个 logit 加一个独立同分布的 Gumbel 噪声 $g_i \sim \text{Gumbel}(0, 1)$，然后取 argmax。可以证明，这样得到的 argmax 严格服从原分布。这看起来像数学小聪明，但它有个大用处：它把「采样的随机性」从 multinomial 抽样挪到了「加噪声后取 argmax」，而 argmax 可以用 softmax 近似，于是**梯度能不间断地流过采样这一步**。这是深度强化学习里 reparameterization 思想的来源。详细推导超出本节范围，可参考 [cs231n note on Gumbel-Softmax](https://cs231n.github.io/) 和原论文 [Gumbel-Softmax](https://arxiv.org/abs/1611.01144)。

## 3. 信息论三件套：怎么衡量「不确定」和「差距」

上面讲了「分布是什么」「怎么采样」。现在要回答两个更细的问题：一个分布自己有多「不确定」？两个分布之间差多少？答案是信息论的三个核心量：entropy、cross-entropy、KL divergence。它们之间有一条 LLM 里最重要的等式。为了避免一开始就堆公式，下面用「公平骰子」和「偏置骰子」两个具体例子来演示。

### 3.1 Entropy：一个分布有多「不确定」

Entropy 衡量一个分布 $p$ 本身有多不确定：$$H(p) = -\sum_x p(x) \log p(x)$$怎么理解？$-\log p(x)$ 是「看到 $x$ 时有多惊讶」——一件小概率的事突然发生了，人会更惊讶。Entropy 就是「惊讶程度的平均」。用 6 面骰子看两个极端：- 公平骰子：每面概率 1/6，完全没法预测，entropy 最大。- 偏置骰子：某一面 99%，其他面加起来 1%，几乎能确定出哪面，entropy 接近 0。对一个有 V 个类别的分布，最大 entropy 是 $\log V$（均匀分布时取到），最小是 0（one-hot 时取到）。LLM 的预测分布 entropy 很高，说明模型很迷茫；很低，说明模型很自信——但自信不一定对。

In [5]:
# 用 6 面骰子对比 entropy
import math

fair_die = torch.tensor([1/6] * 6)
biased_die = torch.tensor([0.99, 0.002, 0.002, 0.002, 0.002, 0.002])
one_hot_die = torch.tensor([1.0, 0.0, 0.0, 0.0, 0.0, 0.0])

def entropy(p, eps=1e-12):
    """计算离散分布的 entropy，加 eps 避免 log(0)"""
    return -(p * torch.log(p + eps)).sum().item()

print(f"公平骰子 entropy:  {entropy(fair_die):.4f}  (最大值 log(6) = {math.log(6):.4f})")
print(f"偏置骰子 entropy:  {entropy(biased_die):.4f}  (几乎确定，接近 0)")
print(f"one-hot entropy:   {entropy(one_hot_die):.4f}  (完全确定，等于 0)")
print()
print("关键观察：uniform 的 entropy 最大，one-hot 的 entropy 最小（为 0）。")
print("LM 训练时，真实答案的 one-hot 分布 entropy = 0，这一点后面会反复用到。")

公平骰子 entropy:  1.7918  (最大值 log(6) = 1.7918)
偏置骰子 entropy:  0.0721  (几乎确定，接近 0)
one-hot entropy:   -0.0000  (完全确定，等于 0)

关键观察：uniform 的 entropy 最大，one-hot 的 entropy 最小（为 0）。
LM 训练时，真实答案的 one-hot 分布 entropy = 0，这一点后面会反复用到。


### 3.2 Cross-Entropy：用预测去「编码」真实，要花多少

Cross-entropy 衡量「用分布 $q$ 去编码真实分布 $p$ 的事件，平均要花多少 bit」：$$H(p, q) = -\sum_x p(x) \log q(x)$$注意两个分布的角色**不一样**：$p$ 是真实分布，$q$ 是预测分布。$-\log q(x)$ 是「按 $q$ 编码 $x$ 需要的 bit 数」，整个式子按 $p$ 的概率加权平均。如果 $q$ 把大部分概率放在 $p$ 的高概率事件上，cross-entropy 就小；如果 $q$ 把概率放错了地方，cross-entropy 就大。LM 训练里 $p$ 是正确答案（本附录第 4 节会解释是 one-hot），$q$ 是模型预测，最小化 cross-entropy 就是让 $q$ 把概率集中到正确的词上。



### 3.3 KL Divergence：两个分布差多少

KL divergence 衡量两个分布的差异：$$D_{KL}(p \| q) = \sum_x p(x) \log \frac{p(x)}{q(x)}$$它有三个重要性质，值得记住：- **非负**：$D_{KL}(p \| q) \ge 0$，等号当且仅当 $p = q$。- **不对称**：$D_{KL}(p \| q) \ne D_{KL}(q \| p)$，所以它不是距离，是「散度」。- **方向敏感**：$p$ 概率为 0 的位置贡献为 0（约定 $0\log 0 = 0$）；但若 $q$ 概率为 0 而 $p$ 概率非 0，会发散到无穷。RLHF 里用 $D_{KL}(\pi_\theta \| \pi_{ref})$ 限制新策略不能偏离参考策略太远；蒸馏里用 $D_{KL}(p_{teacher} \| p_{student})$ 让学生模仿老师。这两处的 KL 选向都有讲究——方向选错会让训练发散。本附录第 7 节的表格会细讲。



### 3.4 最重要的一条等式：H(p, q) = H(p) + D_KL(p || q)

把上面两个定义展开，就能得到一条 LLM 里处处用到的等式。推导很短，把 KL 的定义拆开：$$D_{KL}(p \| q) = \sum_x p(x) \log \frac{p(x)}{q(x)} = \sum_x p(x) \log p(x) - \sum_x p(x) \log q(x) = -H(p) + H(p, q)$$移项：$$H(p, q) = H(p) + D_{KL}(p \| q)$$一句话：**用 $q$ 编码 $p$ 的代价 = $p$ 本身的不确定性 + $q$ 偏离 $p$ 的程度**。前者是数据本身固有的、压不掉的；后者是模型可以改进的部分。这条等式还解释了为什么 LM 训练用 cross-entropy 而不用 KL——两者差一个常数 $H(p)$，对梯度没有影响，但 cross-entropy 只需要 $\log q$，不用算 $\log p$，更省事。下面用骰子验证。

In [6]:
# 用骰子验证 H(p, q) = H(p) + D_KL(p || q)
def cross_entropy(p, q, eps=1e-12):
    """用 q 编码 p 的代价"""
    return -(p * torch.log(q + eps)).sum().item()

def kl_divergence(p, q, eps=1e-12):
    """D_KL(p || q)"""
    return (p * (torch.log(p + eps) - torch.log(q + eps))).sum().item()

p = torch.tensor([0.4, 0.3, 0.2, 0.1])  # 真实分布
q_good = torch.tensor([0.35, 0.35, 0.2, 0.1])  # 接近 p 的预测
q_bad = torch.tensor([0.1, 0.2, 0.3, 0.4])     # 远离 p 的预测

print("真实分布 p:", p.tolist())
print()

for name, q in [("q_good (接近 p)", q_good), ("q_bad (远离 p)", q_bad)]:
    H_p = entropy(p)
    H_pq = cross_entropy(p, q)
    kl = kl_divergence(p, q)
    lhs = H_pq
    rhs = H_p + kl
    print(f"{name}:")
    print(f"  q             = {[round(x, 2) for x in q.tolist()]}")
    print(f"  H(p)          = {H_p:.4f}")
    print(f"  H(p, q)       = {H_pq:.4f}")
    print(f"  D_KL(p || q)  = {kl:.4f}")
    print(f"  H(p)+D_KL     = {rhs:.4f}  (应等于 H(p,q) = {lhs:.4f})")
    print()

print("关键观察：H(p,q) 和 H(p)+D_KL 在数值上完全相等。")
print("q 越接近 p，D_KL 越小；q = p 时 D_KL = 0，此时 H(p,q) = H(p)。")

真实分布 p: [0.4000000059604645, 0.30000001192092896, 0.20000000298023224, 0.10000000149011612]

q_good (接近 p):
  q             = [0.35, 0.35, 0.2, 0.1]
  H(p)          = 1.2799
  H(p, q)       = 1.2870
  D_KL(p || q)  = 0.0072
  H(p)+D_KL     = 1.2870  (应等于 H(p,q) = 1.2870)

q_bad (远离 p):
  q             = [0.1, 0.2, 0.3, 0.4]
  H(p)          = 1.2799
  H(p, q)       = 1.7363
  D_KL(p || q)  = 0.4564
  H(p)+D_KL     = 1.7363  (应等于 H(p,q) = 1.7363)

关键观察：H(p,q) 和 H(p)+D_KL 在数值上完全相等。
q 越接近 p，D_KL 越小；q = p 时 D_KL = 0，此时 H(p,q) = H(p)。


## 4. 为什么 LM 的 loss 用 cross-entropy

把上一节的等式接到 LM 训练上，会看到一条漂亮的等价链：最大似然 → 最小化负对数似然（NLL）→ 最小化 cross-entropy → 最小化 KL（差一个常数）。这一节把这条链走通。**最大似然**：给定训练语料 $x_1, \ldots, x_T$，我们希望模型给这段序列的概率最大化，也就是最大化 $\prod_t p_\theta(x_t | x_{<t})$。取对数变成求和，再取负号变成最小化负对数似然（NLL）：$$\mathcal{L}_{\text{NLL}} = -\sum_t \log p_\theta(x_t | x_{<t})$$现在把 NLL 写成 cross-entropy 的形式。在每个位置，模型输出分布 $q = p_\theta(\cdot | x_{<t})$。训练时的 teacher forcing 假设真实分布 $p$ 是 one-hot——「下一个词就是训练数据里的那个」，没有别的可能。于是 $p$ 在正确词 $x_t$ 上概率为 1，其余为 0。代入 cross-entropy：$$H(p, q) = -\sum_x p(x) \log q(x) = -\log q(x_t) = -\log p_\theta(x_t | x_{<t})$$所以 NLL = cross-entropy（teacher forcing 下两者完全相等）。再套用核心等式：one-hot 的 $H(p) = 0$，所以 cross-entropy = KL divergence。三个量在 LM 训练里完全相等：$$\mathcal{L}_{\text{NLL}} = H(p, q) = D_{KL}(p \| q)$$这就是为什么训练 loss 一节直接 `F.cross_entropy`——它们在数学上是一回事，只是写法不同。



In [7]:
# 用极小数字验证：NLL = H(p, q) = D_KL(p || q) 当 p 是 one-hot 时
logits = torch.tensor([[2.0, 1.0, 0.5, -0.5]])  # 一个位置，4 类
target_id = 1  # 正确 token

# 方法 1: PyTorch 的 cross_entropy（内部就是 NLL）
ce_loss = F.cross_entropy(logits, torch.tensor([target_id])).item()

# 方法 2: 手算 NLL = -log p(correct)
log_probs = F.log_softmax(logits, dim=-1)
nll = -log_probs[0, target_id].item()

# 方法 3: 构造 one-hot p，算 KL
p_onehot = torch.zeros(4)
p_onehot[target_id] = 1.0
q = F.softmax(logits, dim=-1)[0]
# KL(p || q) = sum p log(p/q)，p 只在 target_id 非 0，所以 = log(1/q[target]) = -log q[target]
kl = (p_onehot * (torch.log(p_onehot + 1e-12) - torch.log(q + 1e-12))).sum().item()

print(f"PyTorch cross_entropy: {ce_loss:.6f}")
print(f"手算 NLL:             {nll:.6f}")
print(f"KL(p || q):           {kl:.6f}")
print()
print("三个数字完全相等——验证了 LM 训练下 NLL = CE = KL。")
print("关键观察：H(p) = 0 因为 p 是 one-hot，所以 CE 和 KL 之间没有常数差。")

PyTorch cross_entropy: 1.514675
手算 NLL:             1.514675
KL(p || q):           1.514675

三个数字完全相等——验证了 LM 训练下 NLL = CE = KL。
关键观察：H(p) = 0 因为 p 是 one-hot，所以 CE 和 KL 之间没有常数差。


## 5. Perplexity：把 CE 翻译成「人话」

Perplexity（PPL）是评测 LM 最常用的指标。定义非常简单：$$\text{PPL} = \exp(\text{CE})$$CE 是模型在测试集上的平均 cross-entropy（自然对数底）。CE 是「每个位置平均有多惊讶」的 bit 数，指数化之后，PPL 表示「模型相当于在多少个候选之间均匀犹豫」。几个参考值：- PPL = 1：模型 100% 确定，零犹豫（完美）。- PPL = V（词表大小）：模型在所有词上等概率瞎猜。- PPL = 10：模型相当于在 10 个候选之间均匀犹豫。PPL 越低模型越确定，但「确定」不等于「对」——只是说它对这个猜测有信心。**一个坑：跨数据集不可比**。PPL 强烈依赖 tokenization。同一个模型，在 BPE 词表 50000 的数据上 PPL 可能是 5，在 unigram 词表 100000 的数据上可能是 3。数字小不代表模型好——可能只是词表更大、每个 token 更粗、更好猜。所以 PPL 只能比较「同一数据集 + 同一 tokenizer」上的模型。

In [8]:
# 用一个简单序列演示 PPL 计算
torch.manual_seed(0)

# 假设词表 V=10，模型在 4 个位置给出的 logits
V = 10
logits = torch.tensor([
    [3.0, 1.0, 0.5, -0.5, 0.0, 0.2, -1.0, 0.1, -0.3, 0.4],  # 位置 0
    [0.1, 4.0, 0.2, -1.0, 0.0, 0.1, 0.0, -0.5, 0.3, -0.2],  # 位置 1（很自信）
    [0.3, 0.2, 0.4, 0.1, 0.5, 0.2, 0.3, 0.4, 0.1, 0.5],     # 位置 2（很迷茫）
    [1.5, 0.5, 2.0, -0.5, 0.0, 0.3, -1.0, 0.2, -0.3, 0.1],  # 位置 3
])
targets = torch.tensor([0, 1, 4, 2])

# 计算 CE
ce = F.cross_entropy(logits, targets).item()
ppl = math.exp(ce)

print(f"词表大小 V = {V}")
print(f"序列长度 T = {len(targets)}")
print(f"平均 CE   = {ce:.4f}")
print(f"PPL       = exp(CE) = {ppl:.4f}")
print()
print(f"参考值：")
print(f"  PPL=1 表示完美预测，PPL={V} 表示瞎猜")
print(f"  当前 PPL={ppl:.2f}，介于两者之间")
print(f"  位置 2 的 logits 几乎均匀，CE 贡献最大；位置 1 自信，CE 贡献最小")
print()

# 逐位置 CE
per_pos_ce = F.cross_entropy(logits, targets, reduction='none')
print("逐位置 CE:", [round(x, 3) for x in per_pos_ce.tolist()])
print("逐位置 PPL:", [round(math.exp(x), 2) for x in per_pos_ce.tolist()])
print()
print("关键观察：PPL = exp(CE)，是 CE 的指数化包装。")
print("跨数据集 PPL 不可比——tokenizer 不同，CE 的尺度就不同。")

词表大小 V = 10
序列长度 T = 4
平均 CE   = 0.9208
PPL       = exp(CE) = 2.5113

参考值：
  PPL=1 表示完美预测，PPL=10 表示瞎猜
  当前 PPL=2.51，介于两者之间
  位置 2 的 logits 几乎均匀，CE 贡献最大；位置 1 自信，CE 贡献最小

逐位置 CE: [0.434, 0.146, 2.113, 0.991]
逐位置 PPL: [1.54, 1.16, 8.27, 2.69]

关键观察：PPL = exp(CE)，是 CE 的指数化包装。
跨数据集 PPL 不可比——tokenizer 不同，CE 的尺度就不同。


## 6. 一个反直觉但极其重要的结论：softmax + CE 的梯度

softmax + cross-entropy 组合起来，梯度形式极其简洁：$$\frac{\partial L}{\partial z} = p - \text{one\_hot}(t)$$$z$ 是 logits，$p$ 是 softmax 输出，$t$ 是正确类别。也就是说：正确位置的梯度是 $p_t - 1$（负的，往下压），其余位置是 $p_i$（正的，往下拉）。直觉推导（严谨版见 [cs231n optimization note](https://cs231n.github.io/optimization-2/)）：1. Loss $L = -\log p_t$，对 $p_t$ 求导得 $-1/p_t$。2. softmax 的 Jacobian 满足 $\partial p_j / \partial z_i = p_j(\delta_{ij} - p_i)$，$\delta_{ij}$ 是 Kronecker delta。3. 链式法则：$\partial L / \partial z_i = \sum_j (\partial L / \partial p_j)(\partial p_j / \partial z_i)$，只有 $j = t$ 项非零。4. 代入化简：$\partial L / \partial z_i = -1/p_t \cdot p_t(\delta_{it} - p_i) = p_i - \delta_{it}$。写成向量就是 $\partial L / \partial z = p - \text{one\_hot}(t)$。为什么这个形式让训练稳定？因为它**有界**——$p_i \in [0, 1]$，所以梯度每个分量都在 $[-1, 1]$ 之间，不会像 sigmoid + MSE 那样梯度消失或爆炸。这是所有分类任务都用 softmax + CE 的原因。PyTorch 的 `F.cross_entropy` 内部把 softmax 和 CE 融合成一个 kernel，不显式构造 $p$，直接算 $z - \text{logsumexp}(z)$ 再取正确位置——数值稳定，梯度也对。

In [9]:
# 验证 softmax + CE 的梯度公式：dL/dz = p - one_hot(t)
logits = torch.tensor([2.0, 1.0, 0.5, -0.5], requires_grad=True)
target_id = 1

# 方法 1: PyTorch 自动求导
loss = F.cross_entropy(logits.unsqueeze(0), torch.tensor([target_id]))
loss.backward()
auto_grad = logits.grad.clone()

# 方法 2: 手算 p - one_hot(t)
with torch.no_grad():
    p = F.softmax(logits, dim=-1)
    one_hot = torch.zeros(4)
    one_hot[target_id] = 1.0
    manual_grad = p - one_hot

print(f"target_id = {target_id}")
print(f"softmax 输出 p:  {[round(x, 4) for x in p.tolist()]}")
print(f"one_hot(t):     {one_hot.tolist()}")
print()
print(f"PyTorch 自动梯度: {[round(x, 4) for x in auto_grad.tolist()]}")
print(f"手算 p-one_hot:   {[round(x, 4) for x in manual_grad.tolist()]}")
print()
print("两者完全相等，验证了 dL/dz = p - one_hot(t)。")
print()
print("梯度分量的符号解释：")
for i in range(4):
    g = manual_grad[i].item()
    if i == target_id:
        print(f"  位置 {i}（正确）: 梯度 {g:+.4f} → 负的，往下压（让 z 变小，等价让 p 变大）")
    else:
        print(f"  位置 {i}（错误）: 梯度 {g:+.4f} → 正的，往下拉（让 z 变小，等价让 p 变小）")
print()
print("关键观察：所有梯度分量都在 [-1, 1]，这个有界性是 softmax+CE 训练稳定的原因。")

target_id = 1
softmax 输出 p:  [0.5977, 0.2199, 0.1334, 0.0491]
one_hot(t):     [0.0, 1.0, 0.0, 0.0]

PyTorch 自动梯度: [0.5977, -0.7801, 0.1334, 0.0491]
手算 p-one_hot:   [0.5977, -0.7801, 0.1334, 0.0491]

两者完全相等，验证了 dL/dz = p - one_hot(t)。

梯度分量的符号解释：
  位置 0（错误）: 梯度 +0.5977 → 正的，往下拉（让 z 变小，等价让 p 变小）
  位置 1（正确）: 梯度 -0.7801 → 负的，往下压（让 z 变小，等价让 p 变大）
  位置 2（错误）: 梯度 +0.1334 → 正的，往下拉（让 z 变小，等价让 p 变小）
  位置 3（错误）: 梯度 +0.0491 → 正的，往下拉（让 z 变小，等价让 p 变小）

关键观察：所有梯度分量都在 [-1, 1]，这个有界性是 softmax+CE 训练稳定的原因。


## 7. 一张表：这套数学在 LLM 工程里怎么复用

前面六节建立了词汇表。这一节把它们映射回各个子领域，让你看到训练、RLHF、DPO、蒸馏、评测、推理、MoE 都在操作同一组数学对象。这张表是本附录的核心价值——读完它，再读任何一篇 LLM 论文，都能在上面找到对应的数学结构。| 子领域 | 用到的核心概念 | 数学形式 | 直觉 ||:---|:---|:---|:---|| 预训练 / SFT loss | cross-entropy（= NLL = KL） | $-\log p_\theta(x_t \| x_{<t})$ | 让正确词概率最大 || 评测 | perplexity | $\exp(\text{CE})$ | 每个位置的有效候选数 || 推理：temperature | softmax 温度 | $\text{softmax}(z/T)$ | T→0 greedy，T→∞ uniform || 推理：top-k / top-p | categorical 截断 + 重归一化 | $p^{(S)} / \sum_{j \in S} p_j$ | 限制候选集合 || RLHF（PPO） | KL penalty | $\beta \cdot D_{KL}(\pi_\theta \| \pi_{ref})$ | 防止策略漂移、reward hacking || DPO | log-ratio（隐式 KL） | $\beta \log \frac{\pi_\theta}{\pi_{ref}}$ | 把偏好化成分类 || 蒸馏 | KL（forward 或 reverse） | $D_{KL}(p_{teacher} \| p_{student})$ | 学生模仿老师的分布 || MoE 路由 | categorical 分布 | $\text{softmax}(W_{gate} x)$ | 给每个 token 分配专家概率 || MoE load balancing | entropy / KL 辅助 loss | $\alpha \cdot \text{aux}(\bar{p})$ | 防止专家分配不均 |几个容易混淆的点单独拎出来：- **RLHF 的 KL 选向是 $D_{KL}(\pi_\theta \| \pi_{ref})$**（forward）。原因是 forward KL 在 $\pi_{ref}$ 概率为 0 的位置不会无穷大（因为 $\pi_\theta$ 概率小，乘积趋 0），所以允许新策略探索参考策略没见过的情况；反过来用 reverse KL，新策略会被死死限制在参考策略的支撑集里，无法探索。- **蒸馏通常用 $D_{KL}(p_{teacher} \| p_{student})$**（forward），让学生覆盖老师所有可能输出的位置。也有用 reverse KL 的变体，行为不同。- **DPO 没有显式 KL，但 log-ratio $\log(\pi_\theta / \pi_{ref})$ 是隐式的 KL 项**。它衡量「新策略相对参考策略对某个回答的概率变化」，等价于在偏好空间里做 KL-constrained optimization。- **MoE 的 load balancing loss 形式多样**，有的用 entropy（鼓励 router 分布均匀），有的用 KL（对齐 uniform），有的用「专家利用率方差」。底层都是对 categorical 分布形状的约束。记住这张表，以后读任何 LLM 论文，先问一句：「它在操作哪个分布？在最小化哪个散度？」——绝大多数论文都能用这两句话概括。

## 8. MoE 里的概率论：routing 和 load balancing

MoE 是 categorical 分布和 KL 散度在 LLM 工程里最典型的应用。这一节把本附录第 7 节表格里 MoE 相关的两行展开。MoE 一节讲过 MoE layer 的结构：每个 token 经过一个 router（线性层），输出 num_experts 个分数；取 top-k 个分数最高的专家，按 softmax 权重混合它们的输出。从概率论看，router 输出的就是一个 categorical distribution——「这个 token 被分给每个专家的概率」。**router 的 softmax 是 categorical distribution**。这和 next-token prediction 完全同构：LM head 输出词表上的 categorical，router 输出专家集合上的 categorical。数学结构一致，只是类别含义不同。**Load balancing loss 的概率论形式**。如果 router 自由训练，容易出现「少数专家被疯狂选中、其他闲置」——MoE 退化成 dense model，失去稀疏性优势。解决办法是加一个 auxiliary loss，惩罚专家分配不均。几种常见形式：1. **Entropy 形式**：$L_{aux} = -H(\bar{p})$，$\bar{p}$ 是 batch 内所有 token 的平均路由分布。Entropy 越大分布越均匀，最小化 $-H$ 就是鼓励均匀分配。等价于「让 batch 平均来看，每个专家被选中的概率接近 1/num_experts」。2. **KL 形式**：$L_{aux} = D_{KL}(\bar{p} \| u)$，$u$ 是 uniform。比 entropy 更直接——明确指定目标分布是 uniform。3. **Switch Transformer 形式**：$L_{aux} = N \cdot \sum_i f_i \cdot p_i$，$f_i$ 是专家 i 被选中的频率，$p_i$ 是平均路由概率。看起来不像 KL，但展开后等价于鼓励 $f_i$ 和 $p_i$ 都接近 $1/N$。无论哪种形式，本质都是同一个操作：把 router 输出的 categorical 分布形状约束成 uniform。这是 categorical 分布形状控制的一个典型工程应用。**为什么 router 用 softmax 而不是 sigmoid**？因为 top-k 要求「每个 token 选 k 个互斥专家」，这是 categorical 分布的截断采样；sigmoid 输出的是「每个专家独立选或不选」，语义不同。



In [10]:
# 用一个极小 MoE 演示 router 输出的 categorical 分布
import torch
import torch.nn.functional as F

torch.manual_seed(42)

d_model = 8
num_experts = 4
top_k = 2

# 极简 router：一个线性层
W_gate = torch.randn(d_model, num_experts)

# 模拟 3 个 token
tokens = torch.randn(3, d_model)
gate_logits = tokens @ W_gate  # [3, num_experts]
gate_probs = F.softmax(gate_logits, dim=-1)

print("=== Router 输出的 categorical 分布 ===")
for i in range(3):
    print(f"token {i}: gate_logits = {[round(x, 3) for x in gate_logits[i].tolist()]}")
    print(f"         gate_probs   = {[round(x, 3) for x in gate_probs[i].tolist()]}")
    topk_vals, topk_idx = torch.topk(gate_logits[i], top_k)
    topk_weights = F.softmax(topk_vals, dim=-1)
    print(f"         top-{top_k} 选择: 专家 {topk_idx.tolist()}, 权重 {[round(x, 3) for x in topk_weights.tolist()]}")
    print()

# 计算 batch 平均路由分布，演示 load balancing loss
mean_routing = gate_probs.mean(dim=0)  # [num_experts]
uniform = torch.ones(num_experts) / num_experts

# 形式 1: 负 entropy
neg_entropy = -(mean_routing * torch.log(mean_routing + 1e-12)).sum().item()

# 形式 2: KL(mean_p || uniform)
kl_to_uniform = (mean_routing * (torch.log(mean_routing + 1e-12) - torch.log(uniform + 1e-12))).sum().item()

print("=== Load balancing loss 的两种形式 ===")
print(f"batch 平均路由分布: {[round(x, 4) for x in mean_routing.tolist()]}")
print(f"uniform 分布:       {[round(x, 4) for x in uniform.tolist()]}")
print(f"-H(平均分布)        = {neg_entropy:.4f}  (越小越接近 uniform)")
print(f"KL(平均 || uniform) = {kl_to_uniform:.4f}  (越小越接近 uniform)")
print()
print("关键观察：两种 loss 形式衡量的都是「平均路由分布偏离 uniform 的程度」。")
print("最小化它们都会鼓励 router 把 token 均匀分给所有专家，防止少数专家过载。")

=== Router 输出的 categorical 分布 ===
token 0: gate_logits = [-2.925, -2.457, -2.559, 5.105]
         gate_probs   = [0.0, 0.001, 0.0, 0.999]
         top-2 选择: 专家 [3, 1], 权重 [0.999, 0.001]

token 1: gate_logits = [-1.373, 4.698, -0.633, 2.744]
         gate_probs   = [0.002, 0.87, 0.004, 0.123]
         top-2 选择: 专家 [1, 3], 权重 [0.876, 0.124]

token 2: gate_logits = [-0.313, -4.937, -1.321, 3.506]
         gate_probs   = [0.021, 0.0, 0.008, 0.971]
         top-2 选择: 专家 [3, 0], 权重 [0.979, 0.021]

=== Load balancing loss 的两种形式 ===
batch 平均路由分布: [0.0079, 0.2904, 0.0042, 0.6976]
uniform 分布:       [0.25, 0.25, 0.25, 0.25]
-H(平均分布)        = 0.6712  (越小越接近 uniform)
KL(平均 || uniform) = 0.7151  (越小越接近 uniform)

关键观察：两种 loss 形式衡量的都是「平均路由分布偏离 uniform 的程度」。
最小化它们都会鼓励 router 把 token 均匀分给所有专家，防止少数专家过载。


## 小结

这一节所学的内容：确认你已经理解下面这些点：- [ ] LLM 在每个位置输出的是一个 categorical 分布，softmax 把 logits 变成合法概率- [ ] 数值稳定的 softmax 用 logsumexp 技巧（减去 max），这是 FlashAttention 里 online softmax 的出发点- [ ] temperature 是对 logits 除以 T，T→0 等价 greedy，T→∞ 等价 uniform- [ ] top-k 是固定数量的截断 + 重归一化，top-p 是按累积概率截断 + 重归一化- [ ] entropy 衡量分布本身的不确定性，uniform 时最大，one-hot 时为 0- [ ] cross-entropy 衡量「用 q 编码 p 的代价」，KL 衡量「两个分布的差异」- [ ] 核心等式 $H(p, q) = H(p) + D_{KL}(p \| q)$ 把三者串起来- [ ] teacher forcing 下真实分布是 one-hot，$H(p) = 0$，所以 NLL = CE = KL- [ ] PPL = exp(CE)，含义是「每个位置的有效候选数」；跨数据集不可比- [ ] softmax + CE 的梯度形式极其简洁：$\partial L / \partial z = p - \text{one\_hot}(t)$，有界让训练稳定- [ ] 复用地图：训练 / RLHF / DPO / 蒸馏 / 评测 / 推理 / MoE 都在操作同一组数学对象- [ ] MoE 的 router 输出是 categorical 分布，load balancing loss 是对分布形状的约束（entropy / KL 形式）一句话总结：LLM 工程里反复出现的就是 categorical 分布以及围绕它的信息论三件套（entropy、cross-entropy、KL），抓住这组数学对象，训练、推理、对齐、评测就在同一张地图上对齐了。

## 作业

> 可以用 AI 询问思路、拆步骤、检查方向，但不建议直接让 AI「做完这道题」。

**作业 1：手算 entropy**

给定分布 $p = [0.5, 0.25, 0.125, 0.125]$，手算 $H(p)$。

小提示：$H(p) = -\sum p_i \log p_i$，用自然对数。

In [ ]:
import math
import torch

p = torch.tensor([0.5, 0.25, 0.125, 0.125])

# TODO: 手算 H(p)
manual_H = None  # 在这里填入手算结果

# 用 PyTorch 验证
torch_H = -(p * torch.log(p)).sum().item()

assert manual_H is not None, "请先手算 manual_H"
assert abs(manual_H - torch_H) < 1e-6, f"答案应为 {torch_H:.6f}，你得到 {manual_H}"

print(f"✅ 作业 1 通过：")
print(f"   H(p) = {manual_H:.4f}")
print(f"   理论最大值 log(4) = {math.log(4):.4f}")
print(f"   p 比 uniform 更确定，所以 entropy 小于 log(4)")

**作业 2：实现数值稳定的 log-softmax**

给定一组 logits（含大数），用 logsumexp 技巧实现数值稳定的 log-softmax，不能直接调用 `F.log_softmax`。

小提示：$\log p_i = z_i - \text{logsumexp}(z)$，其中 $\text{logsumexp}(z) = c + \log \sum_j e^{z_j - c}$，$c = \max_j z_j$。

In [ ]:
import torch
import torch.nn.functional as F

big_logits = torch.tensor([1000.0, 1001.0, 1002.0, 999.0])

def stable_log_softmax(z):
    """数值稳定的 log-softmax，不直接调用 F.log_softmax"""
    # TODO: 用 logsumexp 技巧实现
    c = z.max()
    logsumexp = None  # 在这里填入 c + log(sum(exp(z - c)))
    log_probs = None  # 在这里填入 z - logsumexp
    return log_probs

manual_lp = stable_log_softmax(big_logits)
torch_lp = F.log_softmax(big_logits, dim=-1)

assert manual_lp is not None, "请先实现 stable_log_softmax"
assert torch.allclose(manual_lp, torch_lp, atol=1e-5), \
    f"结果与 PyTorch 不一致\n你得到:    {manual_lp.tolist()}\nPyTorch: {torch_lp.tolist()}"

print("✅ 作业 2 通过：")
print(f"   log_softmax = {[round(x, 4) for x in manual_lp.tolist()]}")
print(f"   对应概率     = {[round(x, 4) for x in torch.exp(manual_lp).tolist()]}")
print(f"   概率之和     = {torch.exp(manual_lp).sum().item():.6f}")
print("   关键：减去 max 后再 exp，避免 1000 这种大数溢出。")

**作业 3：实测 PPL**

给定一组 logits 和对应 targets，计算平均 CE 和 PPL，并解释 PPL 的含义。

小提示：PPL = exp(平均 CE)，用 `F.cross_entropy` 算 CE，用 `math.exp` 算 PPL。

In [ ]:
import math
import torch
import torch.nn.functional as F

V = 8
logits = torch.tensor([
    [2.0, 1.0, 0.5, -0.5, 0.0, 0.2, -1.0, 0.1],
    [0.1, 0.2, 0.3, 0.1, 0.2, 0.3, 0.1, 0.2],  # 很迷茫
    [5.0, 0.1, 0.0, -1.0, 0.2, -0.5, 0.1, -0.3],  # 很自信
])
targets = torch.tensor([0, 4, 0])

# TODO: 计算平均 CE 和 PPL
ce = None       # 用 F.cross_entropy
ppl = None      # 用 math.exp(ce)

assert ce is not None and ppl is not None, "请先计算 ce 和 ppl"

# 验证
expected_ce = F.cross_entropy(logits, targets).item()
expected_ppl = math.exp(expected_ce)
assert abs(ce - expected_ce) < 1e-6
assert abs(ppl - expected_ppl) < 1e-3

print(f"✅ 作业 3 通过：")
print(f"   平均 CE = {ce:.4f}")
print(f"   PPL    = {ppl:.4f}")
print(f"   词表 V = {V}")
print()
print(f"   解读：")
print(f"   PPL={ppl:.2f} 表示模型在每个位置相当于在约 {ppl:.1f} 个候选之间均匀犹豫")
print(f"   位置 2（logits=5.0）很自信，CE 贡献最小；位置 1（几乎均匀）CE 贡献最大")
print(f"   如果 PPL 接近 V={V}，说明模型几乎在瞎猜；如果接近 1，说明几乎全对")